# NMF — Template para Novo Corpus

Copie este notebook para `notebooks/nmf/03_nmf_<corpus>.ipynb` e substitua os
placeholders marcados com `# <<` para adaptar a um novo corpus.

## Pipeline (5 etapas)
1. **Lematização** — spaCy (`pt_core_news_lg` ou `en_core_web_sm`) via `_helpers.lemmatize_corpus(..., model_key="nmf")`
2. **Vocabulário** — `gensim.Dictionary`, `no_below=5`, `no_above=0.5` (bloco `nmf:`) → BoW
3. **Seleção de K** — grid search `K ∈ [k_min, k_max]`, maximiza coerência **C_v** (`grid_search_k_nmf`)
4. **Modelo final** — `gensim.models.Nmf` com `K*` e 20 passes (`train_nmf`)
5. **Atribuição** — `argmax(θ)` por documento → `nmf_results.csv`

## Saídas

No **run dir** carimbado (`data/output/<corpus>/nmf/<corpus>_<stamp>/`):

- `nmf_results.csv` — `post_id, topic_id, topic_name, topic_prob_distribution`
- `nmf_topics_for_eval.csv` — `topic_id, topic_name, keywords` (comparativo cross-model)
- `nmf_beta.csv` — matriz completa tópico × vocabulário (habilita retrofits sem refit)
- `nmf_wordclouds.png`, `nmf_keywords_barchart.png`, `nmf_docs_por_topico.png`,
  `nmf_topic_category.png` (se houver categoria/data)

No **diretório-base** do modelo (`data/output/<corpus>/nmf/`, ao lado dos caches CSV):

- `nmf_grid_k_cv.png` — figura de investigação (curva C_v × K)

> Os nomes acima são os que o site Quarto referencia — mantenha-os ao adaptar.

## Antes de executar via `nbconvert` — registre o kernel

O `kernelspec` gravado no `.ipynb` é `python3`, que pode resolver para um Python
diferente do `venv` do projeto (Python do sistema, ou venv de outro projeto) e
falhar com `ModuleNotFoundError` mesmo com tudo instalado. Registre o venv uma vez
por máquina e aponte o `nbconvert` para ele:

```powershell
venv\Scripts\python -m ipykernel install --user --name topic-modeling --display-name "Topic Modeling (venv)"

venv\Scripts\python -m jupyter nbconvert --to notebook --execute --inplace `
  "03-topic-modeling/notebooks/nmf/03_nmf_<corpus>.ipynb" `
  --ExecutePreprocessor.kernel_name=topic-modeling --ExecutePreprocessor.timeout=7200
```


In [ ]:
# Descomente se necessário
# !pip install gensim spacy wordcloud -q
# !python -m spacy download pt_core_news_lg -q
# !python -m spacy download en_core_web_sm -q


In [ ]:
# ── path fix: resolve _helpers.py e data/ de qualquer subdiretório ──
import sys as _sys, os as _os
from pathlib import Path as _P
_here = _P().resolve()
_nb_dir = _here if (_here / '_helpers.py').exists() else _here.parent
if str(_nb_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_dir))
_os.chdir(_nb_dir)          # '../data/...' paths ficam corretos
del _here, _nb_dir, _P, _sys, _os
# ─────────────────────────────────────────────────────────────────────
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _helpers import load_params, get_corpus_config, get_column_names, get_seed, load_corpus
from _helpers import lemmatize_corpus
from _helpers import grid_search_k_nmf, train_nmf, extract_topics_keywords, compute_doc_distributions
from _helpers import name_all_topics
from _helpers import compute_coherence_cv, export_results, export_topics_for_eval

# ── CONFIGURAÇÃO — edite aqui para cada corpus ──────────────────────────
CORPUS_ID = "social"           # << nome do corpus no params.yaml
# ────────────────────────────────────────────────────────────────────────

params = load_params()
corpus_id, corpus_cfg = get_corpus_config(params, CORPUS_ID)
cols = get_column_names(corpus_cfg)
SEED = get_seed(params)
np.random.seed(SEED)

LANG     = corpus_cfg.get("language", "pt")     # "pt" ou "en" — usado em lematização/naming
TEXT_COL = cols["text"]
ID_COL   = cols["post_id"]
DATE_COL = corpus_cfg.get("date_column")        # opcional — habilita o cross-tab tópico × mês

subdir     = corpus_cfg.get("subdir", corpus_id)
INPUT_DIR  = Path(f"../data/input/{subdir}")
from _helpers import make_run_output_dir
# Diretorio-base do MODELO: guarda os caches de grid e as figuras de investigacao,
# compartilhados entre runs.
BASE_OUTPUT_DIR = Path(f"../data/output/{subdir}/nmf")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Saida carimbada por execucao: <corpus>_<data>_<hora> (nao sobrescreve runs anteriores)
OUTPUT_DIR = make_run_output_dir(BASE_OUTPUT_DIR, corpus_id)

nmf_params   = params["nmf"]
k_min, k_max = nmf_params["k_range"]

print(f"Corpus  : {corpus_id}  |  Idioma: {LANG}")
print(f"K range : [{k_min}, {k_max}]")
print(f"Run dir : {OUTPUT_DIR}")


In [ ]:
df       = load_corpus(str(INPUT_DIR))
docs     = df[TEXT_COL].astype(str).tolist()
post_ids = df[ID_COL].astype(str).tolist() if ID_COL in df.columns \
           else [f"doc_{i:04d}" for i in range(len(docs))]

print(f"Docs carregados: {len(docs)}")
print(f"Exemplo: {docs[0][:120]}")


## Lematização (`_helpers.lemmatize_corpus`)

Tokeniza com spaCy (`pt_core_news_lg` ou `en_core_web_sm`, escolhido automaticamente
por `LANG`), remove stop words, tokens não-alfa e tokens com `len(lemma) <= 2`.
Constrói o `gensim.Dictionary` já com `filter_extremes(no_below, no_above)` aplicado
(parâmetros em `params.yaml > nmf`, lidos via `model_key="nmf"`).


In [ ]:
print(f"Lematizando (spaCy, lang={LANG})...")
t0 = time.time()
tokenized, dictionary = lemmatize_corpus(
    docs, LANG, params, model_key="nmf",
    extra_stopwords=corpus_cfg.get("nmf_stopwords_extra", []),  # << opcional: tokens de ruido especificos do NMF para este corpus
)
print(f"  concluido em {time.time()-t0:.1f}s")
print(f"  vocabulario apos filter_extremes: {len(dictionary):,} tokens")
print(f"  Exemplo: {tokenized[0][:20]}")


In [ ]:
corpus_bow = [dictionary.doc2bow(d) for d in tokenized]
non_empty  = sum(1 for d in corpus_bow if len(d) > 0)
print(f"Docs com BoW nao-vazio: {non_empty}/{len(corpus_bow)}")


## Selecao de K (grid search C_v)

Para cada `K ∈ [k_min, k_max]` treina `gensim.models.Nmf` (10 passes) e calcula a coerencia **C_v**.
**Atencao:** C_v sistematicamente penaliza K alto (Stevens et al. 2012).
Use o K* automatico como ponto de partida, mas valide com inspecao qualitativa.
Diferente do LDA, `Nmf` nao expoe `log_perplexity` — nao ha metrica de perplexidade complementar aqui.


In [ ]:
print(f"Grid search K de {k_min} a {k_max} (passes=10)...")
t0 = time.time()
scores = grid_search_k_nmf(
    corpus_bow, dictionary, tokenized, range(k_min, k_max + 1), seed=SEED, passes=10)
print(f"  concluido em {time.time()-t0:.0f}s")

for k in sorted(scores):
    print(f"  K={k:3d}  c_v={scores[k]:.4f}")

best_k = max(scores, key=scores.get)
print(f"\nK* automatico (max c_v): {best_k}  (c_v={scores[best_k]:.4f})")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ks = sorted(scores)
ax.plot(ks, [scores[k] for k in ks], marker="o", linewidth=2, color="steelblue")
ax.axvline(best_k, color="red", linestyle="--", label=f"K*={best_k}")
ax.set_xlabel("K")
ax.set_ylabel("Coerencia C_v")
ax.set_title(f"Grid search K — {corpus_id}")
ax.legend()
plt.tight_layout()
# Figura de INVESTIGACAO: vai para o diretorio-base (acompanha o cache do grid),
# nao para o run dir — o site referencia este path fixo.
plt.savefig(BASE_OUTPUT_DIR / "nmf_grid_k_cv.png", dpi=150, bbox_inches="tight", facecolor="white")
print(f"Salvo: {BASE_OUTPUT_DIR / 'nmf_grid_k_cv.png'}")
plt.show()


## Modelo final

Ajuste `BEST_K` se a inspecao qualitativa sugerir um K diferente do automatico.


In [ ]:
BEST_K = best_k  # << altere se necessario apos inspecao qualitativa
top_n  = params["evaluation"]["top_n_keywords"]

print(f"Treinando NMF final com K={BEST_K}, 20 passes...")
t0 = time.time()
model = train_nmf(corpus_bow, dictionary, k=BEST_K, seed=SEED, passes=20)
print(f"  concluido em {time.time()-t0:.1f}s")

topics_keywords = extract_topics_keywords(model, k=BEST_K, top_n=top_n)
for tid, kws in topics_keywords.items():
    print(f"  T{tid:02d}: {', '.join(kws[:8])}")


In [ ]:
try:
    from wordcloud import WordCloud
    ncols = 3
    nrows = (BEST_K + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
    axes = axes.flatten()
    for i in range(BEST_K):
        ws = dict(model.show_topic(i, topn=30))
        wc = WordCloud(width=400, height=200, background_color="white",
                       max_words=25, colormap="Blues").generate_from_frequencies(ws)
        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].axis("off")
        axes[i].set_title(f"T{i}", fontsize=10)
    for j in range(BEST_K, len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Wordclouds por topico — {corpus_id}  (K={BEST_K})", y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "nmf_wordclouds.png", dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Salvo: {OUTPUT_DIR / 'nmf_wordclouds.png'}")
    plt.show()
except ImportError:
    print("wordcloud nao instalado — pip install wordcloud")


In [ ]:
# Bar chart top-10 keywords por topico (mesma figura dos notebooks de producao)
ncols = 3
nrows = (BEST_K + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows))
axes = axes.flatten() if BEST_K > 1 else [axes]

for tid in range(BEST_K):
    pairs = model.show_topic(tid, topn=10)          # [(palavra, peso em H), ...]
    words = [w for w, _ in pairs][::-1]
    weights = [p for _, p in pairs][::-1]
    axes[tid].barh(words, weights, color="steelblue")
    axes[tid].set_title(f"T{tid}", fontsize=11, fontweight="bold")
    axes[tid].set_xlabel("peso (matriz H)")
    axes[tid].tick_params(labelsize=9)

for j in range(BEST_K, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nmf_keywords_barchart.png", dpi=150, bbox_inches="tight", facecolor="white")
print(f"Salvo: {OUTPUT_DIR / 'nmf_keywords_barchart.png'}")
plt.show()


In [ ]:
llm_cfg = params.get("llm", {})
print(f"Nomeando {BEST_K} tópicos via {llm_cfg.get('model')} (lang={LANG})...")
t0 = time.time()
try:
    topics_names = name_all_topics(
        topics_keywords,
        model=llm_cfg["model"],
        base_url=llm_cfg.get("base_url", "http://localhost:11434"),
        lang=LANG,
    )
    print(f"done em {time.time()-t0:.0f}s")
except Exception as e:
    print(f"LLM falhou ({e}) — fallback top-3 keywords")
    topics_names = {tid: ", ".join(kws[:3]) for tid, kws in topics_keywords.items()}

for tid, name in topics_names.items():
    print(f"  T{tid:02d}: {name}")


In [ ]:
dominant, full = compute_doc_distributions(model, corpus_bow, k=BEST_K)

counts = pd.Series(dominant).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_xlabel("Topico")
ax.set_ylabel("Documentos")
ax.set_title(f"Distribuicao docs por topico — {corpus_id}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nmf_docs_por_topico.png", dpi=150, bbox_inches="tight", facecolor="white")
print(f"Salvo: {OUTPUT_DIR / 'nmf_docs_por_topico.png'}")
plt.show()

cv_final = compute_coherence_cv(topics_keywords, tokenized, dictionary)
print(f"C_v final (K={BEST_K}): {cv_final:.4f}")


In [ ]:
# Cross-tab topico x categoria — validacao EXTERNA (o modelo nao viu esse rotulo).
# Usa a coluna 'category' se existir; senao deriva o mes (YYYY-MM) da coluna de data
# declarada no params.yaml (date_column). Sem nenhuma das duas, a celula se pula.
df_topic = df.copy()
if 'category' not in df_topic.columns and DATE_COL and DATE_COL in df_topic.columns:
    df_topic['category'] = pd.to_datetime(df_topic[DATE_COL], errors='coerce').dt.to_period('M').astype(str)

if 'category' in df_topic.columns:
    import seaborn as sns
    df_topic = df_topic[['category']].assign(topic_id=dominant)
    df_topic = df_topic[df_topic['topic_id'] != -1]
    pivot = df_topic.groupby(['category', 'topic_id']).size().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 0.8), max(4, len(pivot) * 0.5)))
    sns.heatmap(pivot_pct.round(1), annot=True, fmt='.1f', cmap='Blues', linewidths=0.5, ax=ax)
    ax.set_title(f'% documentos por categoria x topico — {corpus_id} (NMF)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'nmf_topic_category.png', dpi=150, bbox_inches="tight", facecolor="white")
    print(f"Salvo: {OUTPUT_DIR / 'nmf_topic_category.png'}")
    plt.show()
else:
    print('Sem coluna category/date_column — cross-tab pulado')


In [ ]:
export_results(
    topics=dominant, probs=full, names=topics_names, texts=docs,
    output_path=str(OUTPUT_DIR / "nmf_results.csv"),
    topic_type="probabilistic", granularity="unit", post_ids=post_ids,
)
export_topics_for_eval(
    topics_keywords=topics_keywords, topics_names=topics_names,
    model_name="nmf", output_path=str(OUTPUT_DIR / "nmf_topics_for_eval.csv"),
)

print(f"Exportado: {OUTPUT_DIR / 'nmf_results.csv'}  ({len(docs)} docs)")
print(f"Exportado: {OUTPUT_DIR / 'nmf_topics_for_eval.csv'}")

# Matriz beta completa (topico x vocabulario) — habilita retrofits sem refit
_beta = model.get_topics()  # (K, vocab_size)
_vocab = [dictionary[i] for i in range(len(dictionary))]
pd.DataFrame(_beta, columns=_vocab).to_csv(OUTPUT_DIR / "nmf_beta.csv", index_label="topic_id")
print(f"Salvo: {OUTPUT_DIR / 'nmf_beta.csv'} ({_beta.shape[0]}x{_beta.shape[1]})")
